In [16]:
# Eval Regression – q3 vs IQR/RMS, residuals by energy, IQR vs training samples (bin 2)
# Plots saved as PDFs in out/regression_eval

import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
if str(ROOT / "notebooks") not in sys.path:
    sys.path.insert(0, str(ROOT / "notebooks"))

CKPT_DIR = Path("/global/cfs/cdirs/m3246/gregork/checkpoints")
OUT_DIR = ROOT / "out" / "regression_eval"
OUT_DIR.mkdir(parents=True, exist_ok=True)

WANDB_TAG = "Run_1203"  # wandb tag to select runs (set as needed)

In [17]:
# Build model -> runs for full dataset (cap=-1) and for scaling (all caps)
from src.utils.utils import get_runs_by_model_and_cap

runs_by_model_cap = get_runs_by_model_and_cap(WANDB_TAG)

# Full-dataset runs: one run per model (cap=-1)
training_names_full = {"Log1p": {}}
baseline_run = None
for model, caps in runs_by_model_cap.items():
    if -1 in caps and caps[-1]:
        run = caps[-1][0]
        training_names_full["Log1p"][model] = run
        if baseline_run is None:
            baseline_run = run

if not training_names_full["Log1p"]:
    raise SystemExit("No runs with cap=-1 found. Check WANDB_TAG and wandb.")
if baseline_run is None:
    baseline_run = list(training_names_full["Log1p"].values())[0]
print("Full-dataset models:", list(training_names_full["Log1p"].keys()))
print("Baseline run for filters/baselines:", baseline_run)

Calling wandb API...


  Fetching runs from fcc_ml/minerva-models with tag 'Run_1203'...
  Wandb API finished: 4 run(s) found with tag 'Run_1203'.
Full-dataset models: ['OLS', 'Transformer1']
Baseline run for filters/baselines: Run_1203_OLS_regression_-1_seed42_20260312_201942


In [18]:
runs_by_model_cap, training_names_full

({'OLS': {-1: ['Run_1203_OLS_regression_-1_seed42_20260312_201942']},
  'Transformer1': {-1: ['Run_1203_regression_Transformer1_data_cap_-1_seed_42_20260312_223412']}},
 {'Log1p': {'OLS': 'Run_1203_OLS_regression_-1_seed42_20260312_201942',
   'Transformer1': 'Run_1203_regression_Transformer1_data_cap_-1_seed_42_20260312_223412'}})

## (I) q3 vs IQR and RMS – full training dataset

In [ ]:
from eval_E_available_plots import plot_rms_iqr

fig_i = plot_rms_iqr(
    CKPT_DIR=CKPT_DIR,
    training_names=training_names_full,
    playlists=["1A"],
    dataset_to_plot="1A",
    baseline_run=baseline_run,
)
fig_i.savefig(OUT_DIR / "q3_vs_iqr_rms_full.pdf", bbox_inches="tight")
print("Saved:", OUT_DIR / "q3_vs_iqr_rms_full_1A.pdf")

No settings found for Run_1203_OLS_regression_-1_seed42_20260312_201942 on playlist 1A
No settings found for Run_1203_regression_Transformer1_data_cap_-1_seed_42_20260312_223412 on playlist 1A
keys:  dict_keys(['1B', '1A'])


In [ ]:
from eval_E_available_plots import plot_rms_iqr

fig_i = plot_rms_iqr(
    CKPT_DIR=CKPT_DIR,
    training_names=training_names_full,
    playlists=["1B"],
    dataset_to_plot="1B",
    baseline_run=baseline_run,
)
fig_i.savefig(OUT_DIR / "q3_vs_iqr_rms_full.pdf", bbox_inches="tight")
print("Saved:", OUT_DIR / "q3_vs_iqr_rms_full_1B.pdf")

In [ ]:

from eval_E_available_plots import plot_rms_iqr

# Plot the same but both on 1A (dashed) and 1B (full)
fig_both = plot_rms_iqr(
    CKPT_DIR=CKPT_DIR,
    training_names=training_names_full,
    playlists=["1A", "1B"],
    dataset_to_plot=["1A", "1B"],
    dataset_to_linestyle={"1A": "--", "1B": "-"},
    baseline_run=baseline_run,
)
fig_both.savefig(OUT_DIR / "q3_vs_iqr_rms_full_1A_1B.pdf", bbox_inches="tight")
print("Saved:", OUT_DIR / "q3_vs_iqr_rms_full_1A_1B.pdf")




## (II) Residuals by energy – full training dataset

In [ ]:
from eval_E_available_plots import plot_residuals_by_energy

fig_ii = plot_residuals_by_energy(
    CKPT_DIR=CKPT_DIR,
    training_names=training_names_full,
    playlists=["1A"],
    dataset_to_plot="1A",
    baseline_run=baseline_run,
)
fig_ii.savefig(OUT_DIR / "residuals_by_energy_full.pdf", bbox_inches="tight")
print("Saved:", OUT_DIR / "residuals_by_energy_full.pdf")



## (III) IQR vs number of training samples – bin 2 (all models)

In [ ]:
# (III) IQR vs number of training samples
# training_names_grouped is constructed inside the plotting cell below.

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
from eval_E_available_plots import plot_rms_iqr_with_uncertainty

# Build grouped training names for this plot on the fly

def _cap_to_label(cap):
    if cap == -1:
        return "6M"
    if cap >= 1_000_000:
        return f"{cap // 1_000_000}M"
    return f"{cap // 1000}k"

training_names_grouped = {"Log1p": {}}
for model, caps in runs_by_model_cap.items():
    for cap, run_list in sorted(caps.items(), key=lambda x: (x[0] == -1, -x[0] if x[0] > 0 else 0)):
        if not run_list:
            continue
        label = f"{model} {_cap_to_label(cap)}"
        training_names_grouped["Log1p"][label] = run_list

if not training_names_grouped["Log1p"]:
    raise SystemExit("No runs for scaling plot. Check WANDB_TAG.")

_, values = plot_rms_iqr_with_uncertainty(
    CKPT_DIR=CKPT_DIR,
    training_names_grouped=training_names_grouped,
    playlists=["1A"],
    dataset_to_plot="1A",
    baseline_run=baseline_run,
    return_values=True,
)
plt.close()

Q3_BIN_IDX = 2

def _n_samples(label):
    m = re.search(r"(\d+(?:\.\d+)?)\s*([kKmM])\b", label)
    if m:
        return float(m.group(1)) * (1e6 if m.group(2).upper() == "M" else 1e3)
    return 0.0

def _method(label):
    return re.sub(r"\s+\d+(?:\.\d+)?[kKmM]$", "", label)

methods = {}
for loss in values:
    if loss in ("q3_bin_mids", "baseline"):
        continue
    for cfg, v in values[loss].items():
        methods.setdefault(_method(cfg), []).append(
            (_n_samples(cfg), v["iqr_mean"][Q3_BIN_IDX], v["iqr_std"][Q3_BIN_IDX])
        )

for m in methods:
    methods[m].sort()

fig_iii, ax = plt.subplots(figsize=(7, 5))
for method, pts in methods.items():
    xs, ys, yerrs = zip(*pts)
    ax.errorbar(xs, ys, yerr=yerrs, marker="o", capsize=4, label=method)

if "baseline" in values:
    ax.axhline(values["baseline"]["iqr"][Q3_BIN_IDX],
               color="black", linestyle=":", lw=1, label="baseline (E_recoil_CCinc)")

ax.set_xscale("log")
ax.set_xlabel("Number of training samples")
ax.set_ylabel("IQR [GeV]")
q3_mid = values["q3_bin_mids"][Q3_BIN_IDX]
ax.set_title(f"IQR vs Training Samples (q$_3$ bin {Q3_BIN_IDX}, mid = {q3_mid:.2f} GeV)")
ax.legend(fontsize=9)
ax.grid(True)
fig_iii.tight_layout()
fig_iii.savefig(OUT_DIR / "iqr_vs_n_samples_bin2.pdf", bbox_inches="tight")
print("Saved:", OUT_DIR / "iqr_vs_n_samples_bin2.pdf")